# 🦜 Colombian Bird Song — Audio Pre-Processing Pipeline

This notebook prepares the audio dataset for model training. It fetches bird recording metadata directly from [Xeno-Canto](https://xeno-canto.org), filters recordings/species using your project rules, downloads MP3 files to Google Drive, and prepares the next steps for mel-spectrogram conversion.

---

## Pipeline Overview

| Step | Description | Status |
|------|-------------|--------|
| 1 | Setup & Authentication | ✅ |
| 2 | Mount Drive & Configure Paths | ✅ |
| 3 | Fetch Metadata from Xeno-Canto | ✅ |
| 4 | Filter Species & Recordings | ✅ |
| 4.5 | Build Ornithological Cards JSON | ✅ |
| 5 | Download Recordings to Drive | ✅ |
| 6 | Chunk MP3s \u2192 WAV export | 🔜 |
| 7 | Diagnostic visualization + Mel-spectrograms | 🔜 |
| 8 | Export tensors for training | 🔜 |

## Step 1 — Setup & Authentication

Install required packages and enter your Xeno-Canto API key securely in the runtime.

Generate the key from your Xeno-Canto account page: https://xeno-canto.org/account/api

The notebook cannot access your account or generate the key for you.

In [ ]:
# Install required packages, ejecute this cell only once unless you're using a colab notebook
#!pip install -r requirements.txt - quiet

In [ ]:
print("Enter your Xeno-Canto API key.")
print("Generate it from your account page: https://xeno-canto.org/account/api")
print("This notebook cannot access your account or create a key for you.")
XC_API_KEY = input("XC_API_KEY: ").strip()

if not XC_API_KEY:
    print("Warning: no API key provided. Metadata fetching will fail until one is set.")

### Import libraries

In [ ]:
from pathlib import Path          
from google.colab import drive        
import shutil                     
import json                         
import requests                  
import time                  
import re                            
import random                       
from datetime import datetime         
import concurrent.futures             
import librosa                       
import numpy as np                  
import soundfile as sf            
from scipy.signal import butter, sosfilt
import matplotlib.pyplot as plt        
import librosa.display    

## Step 2 — Mount Drive & Configure Paths

Mount Google Drive for audio output and configure a temporary runtime folder for intermediate JSON metadata.

- Metadata files are stored in `/content/data` (temporary, cleared after session).
- Audio files are stored in your Drive output folder.

In [ ]:
# Mount Drive for persistent audio storage
drive.mount('/content/drive')

DRIVE_ROOT_PATH = "/content/drive/MyDrive"
DOWNLOAD_FOLDER_NAME = "bird_songs"

TEMP_DATA_DIR = Path("/content/data")
RAW_DIR = TEMP_DATA_DIR / "raw"
RAW_JSON_PATH = TEMP_DATA_DIR / "colombia_birds_by_species.json"
FILTERED_JSON_PATH = TEMP_DATA_DIR / "colombia_birds_filtered.json"

RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path(DRIVE_ROOT_PATH) / DOWNLOAD_FOLDER_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Temporary metadata dir: {TEMP_DATA_DIR}")
print(f"Raw page dir: {RAW_DIR}")
print(f"Drive output dir: {OUTPUT_DIR}")

## ⚙️ Run Configuration

Set `CLEAN_RUN = True` to delete all data produced by previous runs from Drive before starting.
Set `CLEAN_RUN = False` to keep existing Drive data and resume from where the last run left off.

Folders removed when `CLEAN_RUN = True`:
- `bird_songs/` — downloaded MP3s and the persisted filtered metadata JSON
- `processed_audio/` — 3-second WAV chunks
- `bird_tensors/` — compressed `.npz` tensors and `tensor_index.json`

> ⚠️ This cell must be run **after** Step 2 (paths must be defined).

In [ ]:
# ── Set to True to wipe all Drive output from previous runs ──────────────
CLEAN_RUN = False

PROCESSED_DIR = Path(DRIVE_ROOT_PATH) / "processed_audio"
TENSORS_DIR   = Path(DRIVE_ROOT_PATH) / "bird_tensors"

CLEAN_TARGETS = [
    (OUTPUT_DIR,    "bird_songs (MP3s + metadata)"),
    (PROCESSED_DIR, "processed_audio (WAV chunks)"),
    (TENSORS_DIR,   "bird_tensors (NPZ tensors)"),
]

if CLEAN_RUN:
    for target_path, label in CLEAN_TARGETS:
        if target_path.exists():
            shutil.rmtree(target_path)
            target_path.mkdir(parents=True, exist_ok=True)
            print(f"Cleared : {label} ({target_path})")
        else:
            print(f"Not found (nothing to clear): {label}")
    print("Clean run ready. All previous output has been removed.")
else:
    found = [label for path, label in CLEAN_TARGETS if path.exists()]
    if found:
        print("Resuming with existing Drive data:")
        for label in found:
            print(f"  ✓ {label}")
    else:
        print("No existing Drive data found. Starting fresh (same as CLEAN_RUN = True).")

## Step 3 — Fetch Metadata from Xeno-Canto

Fetch all Colombian bird recordings from the Xeno-Canto API v3 (`cnt:colombia grp:birds`), save raw page responses, validate country/group/coordinates, then aggregate results by species.

In [ ]:
BASE_URL = "https://xeno-canto.org/api/3/recordings"
QUERY = "cnt:colombia grp:birds"
PER_PAGE = 500

COL_LAT_MIN, COL_LAT_MAX = -4.5, 13.0
COL_LNG_MIN, COL_LNG_MAX = -82.0, -66.5


def fetch_page(query: str, page: int) -> dict:
    params = {
        "query": query,
        "key": XC_API_KEY,
        "per_page": PER_PAGE,
        "page": page,
    }
    response = requests.get(
        BASE_URL,
        params=params,
        headers={"Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def save_raw_page(data: dict, page: int) -> None:
    with open(RAW_DIR / f"page_{page:04d}.json", "w", encoding="utf-8") as fh:
        json.dump(data, fh, ensure_ascii=False, indent=2)


def in_colombia_bbox(lat: str, lon: str) -> bool:
    try:
        flat, flon = float(lat), float(lon)
    except (ValueError, TypeError):
        return True
    return (COL_LAT_MIN <= flat <= COL_LAT_MAX) and (COL_LNG_MIN <= flon <= COL_LNG_MAX)


def validate_colombian_birds(recordings: list[dict]) -> list[dict]:
    validated = []
    for r in recordings:
        if r.get("grp", "").lower() != "birds":
            continue
        if r.get("cnt", "").lower() != "colombia":
            continue
        if not in_colombia_bbox(r.get("lat", ""), r.get("lon", "")):
            continue
        validated.append(r)
    return validated


def group_by_species(recordings: list[dict]) -> dict:
    species_map = {}

    for r in recordings:
        genus = r.get("gen", "").strip()
        epithet = r.get("sp", "").strip()
        key = f"{genus} {epithet}".strip()

        if key not in species_map:
            species_map[key] = {
                "scientific_name": key,
                "genus": genus,
                "species": epithet,
                "subspecies_seen": sorted({r.get("ssp", "").strip()} - {""}),
                "english_name": r.get("en", ""),
                "group": r.get("grp", ""),
                "recording_count": 0,
                "recordings": [],
            }

        entry = species_map[key]

        ssp = r.get("ssp", "").strip()
        if ssp and ssp not in entry["subspecies_seen"]:
            entry["subspecies_seen"].append(ssp)

        entry["recording_count"] += 1
        entry["recordings"].append(
            {
                "id": r.get("id", ""),
                "quality": r.get("q", ""),
                "length": r.get("length", ""),
                "file": r.get("file", ""),
                "file_name": r.get("file-name", ""),
                "type": r.get("type", ""),
                "sex": r.get("sex", ""),
                "stage": r.get("stage", ""),
                "method": r.get("method", ""),
                "location": r.get("loc", ""),
                "lat": r.get("lat", ""),
                "lon": r.get("lon", ""),
                "date": r.get("date", ""),
                "recordist": r.get("rec", ""),
                "license": r.get("lic", ""),
                "xc_url": r.get("url", ""),
                "also": r.get("also", []),
                "animal_seen": r.get("animal-seen", ""),
                "remarks": r.get("rmk", ""),
            }
        )

    return species_map


def fetch_all_pages() -> list[dict]:
    page = 1
    print(f"[page {page}] fetching ...")
    data = fetch_page(QUERY, page)

    num_pages = int(data.get("numPages", 1))
    print(f"Total pages: {num_pages}")

    all_recordings = list(data.get("recordings", []))
    save_raw_page(data, page)

    for page in range(2, num_pages + 1):
        print(f"[page {page}/{num_pages}] fetching ...")
        data = fetch_page(QUERY, page)
        recordings = data.get("recordings", [])
        all_recordings.extend(recordings)
        save_raw_page(data, page)

    return all_recordings


print("=" * 60)
print("Xeno-Canto metadata fetch")
print("=" * 60)

if not XC_API_KEY:
    print("No XC_API_KEY set. Run Step 1 again and provide your key.")
else:
    all_recordings = fetch_all_pages()
    print(f"Fetched total recordings: {len(all_recordings)}")

    validated = validate_colombian_birds(all_recordings)
    print(f"Validated recordings: {len(validated)}")
    print(f"Discarded recordings: {len(all_recordings) - len(validated)}")

    species_map = group_by_species(validated)

    output = {
        "query": QUERY,
        "total_recordings": len(validated),
        "total_species": len(species_map),
        "species": dict(sorted(species_map.items())),
    }

    with open(RAW_JSON_PATH, "w", encoding="utf-8") as fh:
        json.dump(output, fh, ensure_ascii=False, indent=2)

    print(f"Grouped metadata saved to: {RAW_JSON_PATH}")
    print(f"Raw API page files saved under: {RAW_DIR}")

## Step 4 — Filter Species & Recordings

Apply project filtering rules to the grouped metadata:

- Keep only quality `A` or `B`
- Keep durations between `3` and `60` seconds
- Keep only species with more than `35` surviving recordings

The filtered output is saved to both `/content/data/colombia_birds_filtered.json` (runtime) and `bird_songs/colombia_birds_filtered.json` on Drive for persistence.

If the Drive copy already exists and its `total_recordings` count matches the freshly-filtered result, the file is **not overwritten** and Step 5 will detect that downloads are already up to date and skip automatically.

In [ ]:
MIN_LENGTH_SEC = 3
MAX_LENGTH_SEC = 60
ACCEPTABLE_QUALITY = frozenset({"A", "B"})
MIN_SPECIES_RECORDINGS = 35


def length_seconds(s: str) -> int:
    parts = [int(p) for p in s.strip().split(":")]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    raise ValueError(f"Unexpected length format: {s!r}")


if not RAW_JSON_PATH.exists():
    print(f"Missing grouped metadata: {RAW_JSON_PATH}")
else:
    with open(RAW_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "species" not in data:
        raise KeyError("Input JSON is missing the 'species' key.")

    new_species = {}
    total_recordings = 0

    for name, info in data["species"].items():
        if info.get("recording_count", 0) < MIN_SPECIES_RECORDINGS:
            continue

        kept = []
        for r in info["recordings"]:
            if not r.get("length"):
                continue

            q = (r.get("quality") or "").strip()
            if q not in ACCEPTABLE_QUALITY:
                continue

            duration = length_seconds(r["length"])
            if not (MIN_LENGTH_SEC <= duration <= MAX_LENGTH_SEC):
                continue

            kept.append(r)

        if len(kept) <= MIN_SPECIES_RECORDINGS:
            continue

        species_out = dict(info)
        species_out["recordings"] = kept
        species_out["recording_count"] = len(kept)
        new_species[name] = species_out
        total_recordings += len(kept)

    for name, species_out in new_species.items():
        assert species_out["recording_count"] > MIN_SPECIES_RECORDINGS, (
            f"Integrity check failed for {name!r}: recording_count={species_out['recording_count']}"
        )

    out_data = {
        "query": data["query"],
        "total_recordings": total_recordings,
        "total_species": len(new_species),
        "species": new_species,
    }

    with open(FILTERED_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=2)

    # ── Persist to Drive for cross-session tracking ───────────────────────
    DRIVE_FILTERED_JSON_PATH = OUTPUT_DIR / "colombia_birds_filtered.json"
    drive_needs_update = True
    if DRIVE_FILTERED_JSON_PATH.exists():
        with open(DRIVE_FILTERED_JSON_PATH, "r", encoding="utf-8") as f_drive:
            drive_data = json.load(f_drive)
        if drive_data.get("total_recordings") == total_recordings:
            drive_needs_update = False

    if drive_needs_update:
        with open(DRIVE_FILTERED_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(out_data, f, ensure_ascii=False, indent=2)
        print(f"Drive metadata updated  : {DRIVE_FILTERED_JSON_PATH}")
    else:
        print(f"Drive metadata unchanged: {DRIVE_FILTERED_JSON_PATH} (skipping overwrite)")

    print(f"Filtered species: {len(new_species)}")
    print(f"Filtered recordings: {total_recordings}")
    print(f"Filtered metadata saved to: {FILTERED_JSON_PATH}")

## Step 4.5 — Build Ornithological Cards JSON

Builds a single JSON file (`bird_card/colombian_birds.json`) saved in **Google Drive** under `bird_songs/bird_card/`.

| Field | Source |
|---|---|
| `inat_sightings` | iNaturalist API v1 (research-grade observations) |
| `description` | Wikipedia ES → iNaturalist/Wikipedia EN → GBIF → auto-generated |
| `conservation_status` | Wikipedia taxobox → iNaturalist → Wikidata P141 → SPARQL |
| `taxonomy` | Wikipedia taxobox (order + family) → GBIF fallback |
| `image_url` | Wikipedia thumbnail → iNaturalist taxon photo |
| `info_url` | Wikipedia ES or EN page URL |

**Resumable**: already-fetched entries are loaded from Drive and skipped.  
**No nulls**: every field has a guaranteed fallback value.

In [ ]:
# ── Output path — lives inside the existing bird_songs Drive folder ───────
BIRD_CARD_DIR = OUTPUT_DIR / "bird_card"
OUTPUT_JSON   = BIRD_CARD_DIR / "colombian_birds.json"
BIRD_CARD_DIR.mkdir(parents=True, exist_ok=True)

# ── Load filtered species list (produced by Step 4) ───────────────────────
if not FILTERED_JSON_PATH.exists():
    raise FileNotFoundError(f"Run Steps 3 & 4 first. Missing: {FILTERED_JSON_PATH}")

with open(FILTERED_JSON_PATH, "r", encoding="utf-8") as f:
    filtered = json.load(f)

species_list = list(filtered["species"].keys())
print(f"Species in filtered set : {len(species_list)}")

# ── Load existing output (resumable) ─────────────────────────────────────
if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        existing = json.load(f)
    if isinstance(existing, dict):
        cards = existing.get("species", {})
    elif isinstance(existing, list):
        cards = {e["scientific_name"]: e for e in existing if "scientific_name" in e}
        print(f"  (converted legacy list → {len(cards)} entries)")
    else:
        cards = {}
    print(f"Resuming — {len(cards)} entries already saved.")
else:
    cards = {}

# ── Config ────────────────────────────────────────────────────────────────
INAT_DELAY   = 1.5
GBIF_DELAY   = 1.0
WIKI_DELAY   = 1.5
MAX_RETRIES  = 5
BACKOFF_BASE = 2.0

HEADERS = {
    "User-Agent": "ColombianBirdCards/1.0 (academic research; contact: researcher@example.com)",
    "Accept":     "application/json",
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# ── IUCN lookup tables ────────────────────────────────────────────────────
IUCN_STATUS_MAP = {
    "LC": {"code": "LC", "label_es": "Preocupación menor",         "label_en": "Least Concern"},
    "NT": {"code": "NT", "label_es": "Casi amenazada",              "label_en": "Near Threatened"},
    "VU": {"code": "VU", "label_es": "Vulnerable",                  "label_en": "Vulnerable"},
    "EN": {"code": "EN", "label_es": "En peligro",                  "label_en": "Endangered"},
    "CR": {"code": "CR", "label_es": "En peligro crítico",          "label_en": "Critically Endangered"},
    "EW": {"code": "EW", "label_es": "Extinta en estado silvestre", "label_en": "Extinct in the Wild"},
    "EX": {"code": "EX", "label_es": "Extinta",                     "label_en": "Extinct"},
    "DD": {"code": "DD", "label_es": "Datos insuficientes",         "label_en": "Data Deficient"},
    "NE": {"code": "NE", "label_es": "No evaluada",                 "label_en": "Not Evaluated"},
}
UNKNOWN_STATUS = IUCN_STATUS_MAP["NE"]

IUCN_QIDS = {
    "Q211005":  "LC",
    "Q719675":  "NT",
    "Q278113":  "VU",
    "Q11394":   "EN",
    "Q219127":  "CR",
    "Q239509":  "EW",
    "Q237350":  "EX",
    "Q3245245": "DD",
}

INAT_STATUS_NAME_MAP = {
    "least concern":         "LC",
    "near threatened":       "NT",
    "vulnerable":            "VU",
    "endangered":            "EN",
    "critically endangered": "CR",
    "extinct in the wild":   "EW",
    "extinct":               "EX",
    "data deficient":        "DD",
}

_wikidata_cache = {}

# ── Utilities ─────────────────────────────────────────────────────────────
def safe_get(url, params=None, label=""):
    for attempt in range(MAX_RETRIES):
        try:
            r = SESSION.get(url, params=params, timeout=20)
            if r.status_code == 429:
                wait = BACKOFF_BASE ** (attempt + 2) + random.uniform(2, 5)
                print(f"\n  [429] {label} — retrying in {wait:.1f}s")
                time.sleep(wait)
                continue
            if r.status_code >= 500:
                wait = BACKOFF_BASE ** attempt + random.uniform(1, 3)
                print(f"\n  [{r.status_code}] {label} — retrying in {wait:.1f}s")
                time.sleep(wait)
                continue
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r
        except requests.exceptions.Timeout:
            time.sleep(BACKOFF_BASE ** attempt + 2)
        except requests.exceptions.ConnectionError:
            wait = BACKOFF_BASE ** attempt + random.uniform(1, 3)
            print(f"\n  [conn error] {label} — retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})")
            SESSION.close()
            SESSION.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            print(f"\n  [error] {label}: {e}")
            time.sleep(BACKOFF_BASE ** attempt)
    print(f"\n  [FAILED] {label}")
    return None


def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\[\[(?:[^\]|]+\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\{\{[^}]+\}\}', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_garbage_description(text):
    if not text:
        return True
    t = text.lower()
    if text.count("|") >= 2:
        return True
    if sum(1 for c in text if c.isupper()) / max(len(text), 1) > 0.3:
        return True
    garbage_patterns = [
        r"parque nacional natural", r"reserva forestal",
        r"área importante para la conservaci", r"ecorregi",
        r"iucn red list", r"least concern$", r"endangered$",
        r"vulnerable$", r"critically endangered$",
        r"\bcites\b", r"appendix [ivi]+", r"^[a-z ,;]+$",
    ]
    for pat in garbage_patterns:
        if re.search(pat, t):
            return True
    if "." not in text:
        return True
    bio_verbs = [
        "is ", "are ", "was ", "has ", "have ", "found ", "breed", "inhabit",
        "feed", "nest", "occur", "range", "measure", "weigh", "plumage",
        "species", "bird", "endemic", "distributed", "known",
        "es un", "es una", "ave ", "pájaro", "especie", "se encuentra",
        "habita", "distribuye", "mide ", "pesa ", "plumaje", "endémic",
        "conocid", "alimenta", "anida", "nidifica", "registr",
    ]
    return not any(v in t for v in bio_verbs)


def save_cards(cards_dict):
    out = {
        "generated_at":  datetime.utcnow().isoformat() + "Z",
        "total_species": len(cards_dict),
        "species":       cards_dict,
    }
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)


# ── Conservation status helpers ───────────────────────────────────────────
def _iucn_from_code(raw_code):
    code = (raw_code or "").strip().upper()
    return IUCN_STATUS_MAP.get(code, UNKNOWN_STATUS)


def _iucn_from_inat_name(status_name):
    key = (status_name or "").strip().lower()
    code = INAT_STATUS_NAME_MAP.get(key)
    return IUCN_STATUS_MAP.get(code, UNKNOWN_STATUS) if code else None


def _wikidata_entity_by_site(page_title, site="enwiki"):
    cache_key = f"{site}:{page_title}"
    if cache_key in _wikidata_cache:
        return _wikidata_cache[cache_key]
    r = safe_get(
        "https://www.wikidata.org/w/api.php",
        params={
            "action": "wbgetentities", "sites": site, "titles": page_title,
            "props": "claims", "format": "json",
        },
        label=f"Wikidata [{site}:{page_title}]",
    )
    if r is None:
        return None
    for qid, entity in r.json().get("entities", {}).items():
        if not qid.startswith("-"):
            _wikidata_cache[cache_key] = entity
            return entity
    return None


def _status_from_wikidata_entity(entity):
    if not entity:
        return None
    claims = entity.get("claims", {})
    if "P141" not in claims:
        return None
    try:
        status_qid = claims["P141"][0]["mainsnak"]["datavalue"]["value"]["id"]
        code = IUCN_QIDS.get(status_qid)
        return IUCN_STATUS_MAP.get(code) if code else None
    except (KeyError, IndexError, TypeError):
        return None


def get_conservation_status(info_url, scientific_name, wiki_iucn_code=""):
    # 1. Wikipedia taxobox (zero extra calls)
    if wiki_iucn_code and wiki_iucn_code in IUCN_STATUS_MAP:
        return IUCN_STATUS_MAP[wiki_iucn_code]

    # 2. Wikidata via Wikipedia title
    titles_to_try = []
    if info_url and "/wiki/" in info_url:
        raw  = requests.utils.unquote(
            info_url.rstrip("/").split("/wiki/")[-1]
        ).replace("_", " ")
        site = "eswiki" if "es.wikipedia.org" in info_url else "enwiki"
        titles_to_try.append((raw, site))
    if scientific_name:
        titles_to_try.append((scientific_name, "enwiki"))
        titles_to_try.append((scientific_name, "eswiki"))

    for title, site in titles_to_try:
        entity = _wikidata_entity_by_site(title, site)
        time.sleep(0.4)
        result = _status_from_wikidata_entity(entity)
        if result:
            return result

    # 3. SPARQL fallback
    if scientific_name:
        try:
            query = f'''
            SELECT ?status WHERE {{
              ?item wdt:P225 "{scientific_name}" .
              ?item wdt:P141 ?status .
            }} LIMIT 1
            '''
            r = safe_get(
                "https://query.wikidata.org/sparql",
                params={"query": query, "format": "json"},
                label=f"SPARQL status [{scientific_name}]",
            )
            time.sleep(0.8)
            if r:
                bindings = r.json().get("results", {}).get("bindings", [])
                if bindings:
                    qid  = bindings[0]["status"]["value"].split("/")[-1]
                    code = IUCN_QIDS.get(qid)
                    if code:
                        return IUCN_STATUS_MAP[code]
        except Exception as e:
            print(f"  [WARN SPARQL] {scientific_name}: {e}")

    return UNKNOWN_STATUS


# ── Data sources ──────────────────────────────────────────────────────────
def inat_taxa(scientific_name):
    result = {"description": "", "image_url": "", "info_url": "", "conservation_status": None}
    r = safe_get(
        "https://api.inaturalist.org/v1/taxa",
        params={"q": scientific_name, "per_page": 1, "rank": "species"},
        label=f"iNat taxa [{scientific_name}]",
    )
    time.sleep(INAT_DELAY)
    if r is None:
        return result

    results = r.json().get("results", [])
    if not results:
        return result

    taxon = results[0]
    wiki_summary = clean_text(taxon.get("wikipedia_summary", "") or "")
    if wiki_summary:
        result["description"] = wiki_summary[:800]
    result["info_url"]  = taxon.get("wikipedia_url", "") or ""
    photo = taxon.get("default_photo") or {}
    result["image_url"] = photo.get("medium_url", "") or photo.get("url", "") or ""

    cs          = taxon.get("conservation_status") or {}
    raw_code    = cs.get("status") or ""
    status_name = taxon.get("conservation_status_name") or ""
    if raw_code:
        status = _iucn_from_code(raw_code)
        if status["code"] != "NE":
            result["conservation_status"] = status
    if result["conservation_status"] is None and status_name:
        result["conservation_status"] = _iucn_from_inat_name(status_name)

    return result


def inat_sightings(scientific_name):
    r = safe_get(
        "https://api.inaturalist.org/v1/observations",
        params={"taxon_name": scientific_name, "quality_grade": "research", "per_page": 0},
        label=f"iNat obs [{scientific_name}]",
    )
    time.sleep(INAT_DELAY)
    return int(r.json().get("total_results", 0)) if r else 0


def gbif_description(scientific_name):
    r = safe_get(
        "https://api.gbif.org/v1/species/match",
        params={"name": scientific_name, "kingdom": "Animalia", "class": "Aves"},
        label=f"GBIF match [{scientific_name}]",
    )
    time.sleep(GBIF_DELAY)
    if r is None:
        return ""
    data = r.json()
    key = data.get("usageKey") or data.get("speciesKey")
    if not key:
        return ""
    r2 = safe_get(
        f"https://api.gbif.org/v1/species/{key}/descriptions",
        label=f"GBIF desc [{scientific_name}]",
    )
    time.sleep(GBIF_DELAY)
    if r2 is None:
        return ""
    best_text, best_score = "", 0
    for d in r2.json().get("results", []):
        lang  = (d.get("language") or "").lower()
        dtype = (d.get("type") or "").lower()
        text  = clean_text(d.get("description") or "")
        if len(text) < 120 or is_garbage_description(text):
            continue
        score = 0
        if lang in ("spa", "es"):    score += 18
        elif lang in ("eng", "en"):  score += 12
        elif lang:                   score -= 5
        if dtype in ("", "general", "description", "biology", "ecology", "diagnostic"):
            score += 8
        if dtype in ("distribution", "conservation status", "threats", "habitat"):
            score -= 5
        score += min(len(text) // 80, 8)
        if score > best_score:
            best_score, best_text = score, text
    return best_text[:800]


def wiki_intro(scientific_name, lang="es"):
    base = f"https://{lang}.wikipedia.org/w/api.php"
    out  = {"description": "", "info_url": "", "image_url": ""}
    r = safe_get(
        base,
        params={"action": "query", "list": "search", "srsearch": scientific_name,
                "srlimit": 3, "format": "json"},
        label=f"Wiki {lang.upper()} search [{scientific_name}]",
    )
    time.sleep(WIKI_DELAY)
    if r is None:
        return out
    results = r.json().get("query", {}).get("search", [])
    if not results:
        return out
    genus   = scientific_name.split()[0].lower()
    epithet = scientific_name.split()[-1].lower()
    title   = next(
        (res["title"] for res in results
         if genus in res["title"].lower() or epithet in res["title"].lower()),
        results[0]["title"],
    )
    r2 = safe_get(
        base,
        params={"action": "query", "titles": title, "prop": "extracts|pageimages",
                "exintro": 1, "explaintext": 1, "pithumbsize": 600, "format": "json"},
        label=f"Wiki {lang.upper()} extract [{title}]",
    )
    time.sleep(WIKI_DELAY)
    safe_title    = title.replace(" ", "_")
    out["info_url"] = f"https://{lang}.wikipedia.org/wiki/{safe_title}"
    if r2 is None:
        return out
    for pg in r2.json().get("query", {}).get("pages", {}).values():
        if pg.get("pageid", -1) == -1:
            break
        extract = clean_text(pg.get("extract", "") or "")
        if extract and not is_garbage_description(extract):
            sentences = re.split(r'(?<=[.!?])\s+', extract)
            out["description"] = " ".join(sentences[:3])[:650]
        thumb = pg.get("thumbnail") or {}
        out["image_url"] = thumb.get("source", "")
    return out


def wiki_taxonomy(page_title):
    result = {"order": "", "family": "", "iucn_code": ""}
    if not page_title:
        return result

    is_es = False
    if page_title.startswith("http"):
        is_es       = "es.wikipedia.org" in page_title
        page_title  = requests.utils.unquote(
            page_title.rstrip("/").split("/")[-1]
        ).replace("_", " ")

    apis = (
        ["https://es.wikipedia.org/w/api.php", "https://en.wikipedia.org/w/api.php"]
        if is_es else
        ["https://en.wikipedia.org/w/api.php", "https://es.wikipedia.org/w/api.php"]
    )

    def extract_field(wikitext, key):
        m = re.search(r'\|\s*' + key + r'\s*=\s*([^\|\}\n]+)', wikitext, re.IGNORECASE)
        if not m:
            return ""
        raw = m.group(1).strip()
        raw = re.sub(r'\[\[(?:[^\]|]+\|)?([^\]]+)\]\]', r'\1', raw)
        raw = re.sub(r'\{\{[^}]+\}\}', '', raw)
        raw = re.sub(r'<[^>]+>', '', raw)
        return raw.strip()

    for api_url in apis:
        r = safe_get(
            api_url,
            params={"action": "query", "titles": page_title, "prop": "revisions",
                    "rvprop": "content", "rvslots": "main", "rvlimit": 1, "format": "json"},
            label=f"Wiki taxobox [{page_title}]",
        )
        time.sleep(WIKI_DELAY)
        if r is None:
            continue
        wikitext = ""
        for pg in r.json().get("query", {}).get("pages", {}).values():
            if pg.get("pageid", -1) == -1:
                continue
            revs = pg.get("revisions", [])
            if revs:
                wikitext = revs[0].get("slots", {}).get("main", {}).get("*", "")
        if not wikitext:
            continue
        order  = extract_field(wikitext, "ordo")   or extract_field(wikitext, "order")
        family = extract_field(wikitext, "familia") or extract_field(wikitext, "family")
        if order or family:
            result["order"], result["family"] = order, family
        for field in ["estado_UICN", "status", "conservation_status"]:
            raw = extract_field(wikitext, field).upper()
            if raw in IUCN_STATUS_MAP:
                result["iucn_code"] = raw
                break
        if result["order"] or result["family"]:
            return result

    # GBIF fallback for order/family
    r = safe_get(
        "https://api.gbif.org/v1/species/match",
        params={"name": page_title, "kingdom": "Animalia", "class": "Aves"},
        label=f"GBIF taxonomy [{page_title}]",
    )
    time.sleep(GBIF_DELAY)
    if r:
        data = r.json()
        result["order"]  = data.get("order", "")  or ""
        result["family"] = data.get("family", "") or ""
        if result["order"] or result["family"]:
            return result

    print(f"  [WARN taxonomy] No order/family found for: {page_title}")
    return result


def build_taxonomy(sci_name, sp_info, extra):
    genus   = sp_info.get("genus", "")   or sci_name.split()[0]
    species = sp_info.get("species", "") or (sci_name.split()[1] if " " in sci_name else "")
    return {
        "kingdom": "Animalia",
        "phylum":  "Chordata",
        "class":   "Aves",
        "order":   extra.get("order",  ""),
        "family":  extra.get("family", ""),
        "genus":   genus,
        "species": species,
    }


def auto_description(sci_name, en_name, tax):
    parts = [f"{en_name} ({sci_name})" if en_name else sci_name, "is a bird species"]
    if tax.get("order"):
        parts.append(f"in the order {tax['order']}")
    if tax.get("family"):
        parts.append(f"(family {tax['family']})")
    parts.append("recorded in Colombia.")
    return " ".join(parts)


# ── Main loop ─────────────────────────────────────────────────────────────
to_fetch = [s for s in species_list if s not in cards]
print(f"Entries to fetch : {len(to_fetch)}  (skipping {len(cards)} already done)")
print(f"Estimated time   : ~{len(to_fetch) * 10 // 60} min\n")

for i, sci_name in enumerate(to_fetch, 1):
    sp_info   = filtered["species"][sci_name]
    en_name   = sp_info.get("english_name", "") or ""
    rec_count = sp_info.get("recording_count", 0)

    print(f"[{i:>3}/{len(to_fetch)}] {sci_name:<38}", end=" ", flush=True)

    # 1. Wikipedia ES
    wiki_es     = wiki_intro(sci_name, lang="es")
    description = wiki_es["description"]
    image_url   = wiki_es["image_url"]
    info_url    = wiki_es["info_url"]
    desc_source = "Wikipedia ES" if description else ""

    # 2. iNaturalist
    inat = inat_taxa(sci_name)
    if not image_url:
        image_url = inat["image_url"]
    if not info_url:
        info_url = inat["info_url"]
    if not description and inat["description"]:
        description = inat["description"]
        desc_source = "iNat/Wikipedia EN"

    # 3. GBIF descriptions
    if not description:
        description = gbif_description(sci_name)
        if description:
            desc_source = "GBIF"

    # 4. Wikipedia EN (last resort)
    if not description or not image_url or not info_url:
        wiki_en = wiki_intro(sci_name, lang="en")
        if not description and wiki_en["description"]:
            description = wiki_en["description"]
            desc_source = "Wikipedia EN"
        if not image_url:
            image_url = wiki_en["image_url"]
        if not info_url:
            info_url = wiki_en["info_url"]

    # Taxonomy + IUCN code from taxobox (single wikitext download)
    taxo_extra = wiki_taxonomy(info_url or sci_name)
    tax        = build_taxonomy(sci_name, sp_info, taxo_extra)

    # iNaturalist sightings
    sightings = inat_sightings(sci_name)

    # Conservation status: iNat → taxobox → Wikidata → SPARQL
    conservation = inat.get("conservation_status")
    if not conservation or conservation["code"] == "NE":
        conservation = get_conservation_status(
            info_url, sci_name,
            wiki_iucn_code=taxo_extra.get("iucn_code", ""),
        )

    # Fallbacks — guarantee no nulls
    if not description:
        description = auto_description(sci_name, en_name, tax)
        desc_source = "auto"
    if not info_url:
        info_url = f"https://en.wikipedia.org/wiki/{sci_name.replace(' ', '_')}"
    if not image_url:
        r_img = safe_get(
            "https://api.inaturalist.org/v1/taxa/autocomplete",
            params={"q": sci_name, "per_page": 1},
            label=f"iNat img fallback [{sci_name}]",
        )
        time.sleep(INAT_DELAY)
        if r_img:
            res = r_img.json().get("results", [])
            if res:
                p = res[0].get("default_photo") or {}
                image_url = p.get("medium_url", "") or p.get("url", "")
        if not image_url:
            image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ac/No_image_available.svg/480px-No_image_available.svg.png"

    cards[sci_name] = {
        "scientific_name":       sci_name,
        "english_name":          en_name,
        "inat_sightings":        sightings,
        "xeno_canto_recordings": rec_count,
        "description":           description,
        "description_source":    desc_source,
        "conservation_status":   conservation,
        "taxonomy":              tax,
        "image_url":             image_url,
        "info_url":              info_url,
    }

    img_ok   = "img=OK" if image_url and "No_image" not in image_url else "img=fallback"
    cons_ok  = conservation["code"] if conservation else "??"
    print(f"iNat={sightings:>6}  status={cons_ok:<3}  src={desc_source:<14}  {img_ok}")

    if i % 5 == 0:
        save_cards(cards)
        print(f"  >>> Saved {len(cards)} entries.\n")

    time.sleep(0.5)

# Mystery Mystery catch-all
if "Mystery Mystery" not in cards:
    cards["Mystery Mystery"] = {
        "scientific_name":       "Unknown / Uncatalogued Species",
        "english_name":          "Unknown / Uncatalogued Species",
        "inat_sightings":        0,
        "xeno_canto_recordings": 0,
        "description": (
            "No se pudo determinar una especie específica para esta grabación, pero se confirmó que es un canto de ave registrado en Colombia."
        ),
        "description_source":  "manual",
        "conservation_status": UNKNOWN_STATUS,
        "taxonomy": {
            "kingdom": "Unknown", "phylum": "Unknown", "class": "Unknown",
            "order": "Unknown", "family": "Unknown",
            "genus": "Unknown", "species": "Unknown",
        },
        "image_url": "",
        "info_url":  "",
    }
    print("Mystery Mystery entry added.")
if "Mystery mystery" not in cards:
    cards["Mystery mystery"] = {
        "scientific_name":       "Unknown / Uncatalogued Species",
        "english_name":          "Unknown / Uncatalogued Species",
        "inat_sightings":        0,
        "xeno_canto_recordings": 0,
        "description": (
            "No se pudo determinar una especie específica para esta grabación, pero se confirmó que es un canto de ave registrado en Colombia."
        ),
        "description_source":  "manual",
        "conservation_status": UNKNOWN_STATUS,
        "taxonomy": {
            "kingdom": "Unknown", "phylum": "Unknown", "class": "Unknown",
            "order": "Unknown", "family": "Unknown",
            "genus": "Unknown", "species": "Unknown",
        },
        "image_url": "",
        "info_url":  "",
    }
    print("Mystery mystery entry added.")
save_cards(cards)
print(f"\nDone. {len(cards)} entries saved to:\n  {OUTPUT_JSON}")

# Integrity check
null_desc = [k for k, v in cards.items() if not v.get("description")]
null_img  = [k for k, v in cards.items() if not v.get("image_url")]
null_url  = [k for k, v in cards.items() if not v.get("info_url")]
ne_status = [k for k, v in cards.items() if v.get("conservation_status", {}).get("code") == "NE"]
by_source = {}
for v in cards.values():
    s = v.get("description_source", "unknown")
    by_source[s] = by_source.get(s, 0) + 1
print(f"   Null descriptions : {len(null_desc)}")
print(f"   Null images       : {len(null_img)}")
print(f"   Null info URLs    : {len(null_url)}")
print(f"   NE status (may be real): {len(ne_status)}")
print(f"   Description sources: {by_source}")
for k in set(null_desc + null_img + null_url):
    print(f"   WARNING missing data: {k}")


## Step 5 — Download Recordings to Google Drive

Load filtered metadata from Drive (`bird_songs/colombia_birds_filtered.json`) if it exists, otherwise fall back to the runtime copy. Compare it against the Drive copy to detect whether the recording list has changed.

- If the Drive metadata matches the runtime-filtered result (`total_recordings` is equal), all expected MP3s are assumed to be present and **this step is skipped automatically**.
- If the metadata differs or no Drive copy exists, download only the missing MP3 files. Existing files are always skipped.

In [ ]:
DRIVE_FILTERED_JSON_PATH = OUTPUT_DIR / "colombia_birds_filtered.json"

# ── Decide which metadata source to use ──────────────────────────────────
if DRIVE_FILTERED_JSON_PATH.exists():
    with open(DRIVE_FILTERED_JSON_PATH, "r", encoding="utf-8") as f:
        drive_data = json.load(f)
    print(f"Loaded Drive metadata: {drive_data.get('total_species', 0)} species, "
          f"{drive_data.get('total_recordings', 0)} recordings.")
else:
    drive_data = None
    print("No Drive metadata found. Will use runtime-filtered metadata.")

if FILTERED_JSON_PATH.exists():
    with open(FILTERED_JSON_PATH, "r", encoding="utf-8") as f:
        runtime_data = json.load(f)
else:
    runtime_data = None

# Use Drive copy if available, else fall back to runtime copy
data = drive_data if drive_data is not None else runtime_data

if data is None:
    print("No filtered metadata available. Run Steps 3 and 4 first.")
else:
    # ── Check if downloads are already up to date ─────────────────────────
    drive_rec_count  = drive_data.get("total_recordings", -1) if drive_data else -1
    runtime_rec_count = runtime_data.get("total_recordings", -2) if runtime_data else -2

    if drive_rec_count == runtime_rec_count and drive_rec_count != -1:
        print(
            f"Drive metadata matches runtime ({drive_rec_count} recordings). "
            "Checking for missing MP3s — skipping Step 5 if all files are present."
        )
        # Count expected vs present files
        expected = [
            OUTPUT_DIR / info.get("scientific_name", name).replace(" ", "_") / f"XC{r['id']}.mp3"
            for name, info in data.get("species", {}).items()
            for r in info.get("recordings", [])
            if r.get("id")
        ]
        missing = [p for p in expected if not p.exists()]
        if not missing:
            print(f"All {len(expected)} MP3s already on Drive. Step 5 skipped.")
            data = None  # signal to skip the download block below

    if data is not None:
        download_tasks = []

        for species_name, species_info in data.get("species", {}).items():
            safe_species_name = species_info.get("scientific_name", species_name).replace(" ", "_")
            species_dir = OUTPUT_DIR / safe_species_name
            species_dir.mkdir(parents=True, exist_ok=True)

            for recording in species_info.get("recordings", []):
                file_url = recording.get("file")
                rec_id   = recording.get("id")
                if file_url and rec_id:
                    out_file = species_dir / f"XC{rec_id}.mp3"
                    download_tasks.append((file_url, out_file))

        print(f"Prepared {len(download_tasks)} files for download to {OUTPUT_DIR}")

        def download_file(url: str, out_path: Path) -> bool:
            if out_path.exists():
                return False
            try:
                if url.startswith("//"):
                    url = "https:" + url
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                with open(out_path, "wb") as f_out:
                    f_out.write(response.content)
                return True
            except Exception:
                return False

        MAX_WORKERS      = 4
        downloaded_count = 0
        skipped_count    = 0

        print("Starting downloads...")
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_item = {
                executor.submit(download_file, url, path): (url, path)
                for url, path in download_tasks
            }
            for index, future in enumerate(concurrent.futures.as_completed(future_to_item), start=1):
                if future.result():
                    downloaded_count += 1
                else:
                    skipped_count += 1
                if index % 100 == 0:
                    print(f"Processed {index}/{len(download_tasks)} items...")

        print(f"Download complete. Downloaded: {downloaded_count}, skipped: {skipped_count}")

## Step 6 — Chunking, Filtering & WAV Export

Load each MP3 from Drive, apply a 5th-order Butterworth band-pass filter (500 Hz – 15 kHz), segment into 3-second chunks with 50% overlap (96 000 samples, 48 000-sample stride), reject silent chunks whose RMS energy is below 2% of the file peak, peak-normalize each surviving chunk to `[-1, 1]`, then write it as a 16-bit `.wav` to `processed_audio/<Species>/` on Drive.

In [ ]:
SR               = 32_000
CHUNK_LEN        = 3.0
CHUNK_SAMPLES    = int(SR * CHUNK_LEN)   # 96 000
STRIDE_SAMPLES   = CHUNK_SAMPLES // 2   # 48 000 (50% overlap)
BUTTER_LOW_HZ    = 500
BUTTER_HIGH_HZ   = 15_000
RMS_RATIO_THRESH = 0.02

SONGS_DIR     = OUTPUT_DIR
PROCESSED_DIR = Path(DRIVE_ROOT_PATH) / "processed_audio"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def make_bandpass_sos(low_hz, high_hz, sr, order=5):
    nyq = sr / 2.0
    return butter(order, [low_hz / nyq, high_hz / nyq], btype="band", output="sos")


def apply_bandpass(audio, sos):
    return sosfilt(sos, audio).astype(np.float32)


def chunk_audio(audio, chunk_len, stride):
    return [audio[s : s + chunk_len].copy()
            for s in range(0, len(audio) - chunk_len + 1, stride)]


def reject_silent(chunks, rms_ratio):
    if not chunks:
        return []
    rms_values = np.array([np.sqrt(np.mean(c ** 2)) for c in chunks])
    threshold  = rms_values.max() * rms_ratio
    return [c for c, r in zip(chunks, rms_values) if r >= threshold]


def peak_normalize(chunk):
    peak = np.abs(chunk).max()
    return (chunk / peak).astype(np.float32) if peak > 0 else chunk


sos                   = make_bandpass_sos(BUTTER_LOW_HZ, BUTTER_HIGH_HZ, SR)
errors                = []
total_files_processed = 0
total_chunks_saved    = 0

species_dirs = sorted(d for d in SONGS_DIR.iterdir() if d.is_dir())
print(f"Processing {len(species_dirs)} species ...")

for species_dir in species_dirs:
    out_dir = PROCESSED_DIR / species_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)

    for mp3_path in sorted(species_dir.glob("*.mp3")):
        total_files_processed += 1
        try:
            audio, _  = librosa.load(str(mp3_path), sr=SR, mono=True)
            filtered  = apply_bandpass(audio, sos)
            chunks    = chunk_audio(filtered, CHUNK_SAMPLES, STRIDE_SAMPLES)
            chunks    = reject_silent(chunks, RMS_RATIO_THRESH)
            for idx, chunk in enumerate(chunks):
                norm     = peak_normalize(chunk)
                out_path = out_dir / f"{mp3_path.stem}_chunk{idx:03d}.wav"
                sf.write(str(out_path), norm, SR, subtype="PCM_16")
                total_chunks_saved += 1
        except Exception as exc:
            errors.append((str(mp3_path), str(exc)))

print(f"Files processed : {total_files_processed}")
print(f"Chunks saved    : {total_chunks_saved}")
print(f"Errors          : {len(errors)}")
if errors:
    for p, m in errors[:10]:
        print(f"  {p}: {m}")

## Step 7 — Diagnostic Visualization for Normalization Selection

Sample up to 300 chunks from `processed_audio/`, compute raw log-Mel spectrograms, and produce four diagnostic plots to inform the normalization choice before tensor export:

1. Side-by-side spectrogram panels (quiet / medium / loud)
2. Global log-Mel dB histogram with key percentiles
3. Per-Mel-bin mean and standard deviation
4. Same chunk under raw dB, fixed-range `[0, 1]`, and per-band z-score

**Use the plots to decide `NORM_METHOD` in Step 8.** All statistics are computed from the training split only.

In [ ]:
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 128
SR         = 32_000
DB_MIN     = -80.0
DB_MAX     = 0.0

DIAG_SAMPLE_SIZE = 300
random.seed(42)

all_wav_paths = sorted(PROCESSED_DIR.rglob("*.wav"))
sampled_paths = random.sample(all_wav_paths, min(DIAG_SAMPLE_SIZE, len(all_wav_paths)))
print(f"Sampled {len(sampled_paths)} chunks from {len(all_wav_paths)} total.")


def wav_to_logmel(path):
    audio, _ = librosa.load(str(path), sr=SR, mono=True)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    return librosa.power_to_db(mel, ref=np.max)


# ── Plot 1: Spectrogram panels ─────────────────────────────────────────────
panel_dbs = [(p, wav_to_logmel(p)) for p in sampled_paths[:50]]
panel_dbs.sort(key=lambda x: x[1].max())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Log-Mel Spectrograms: quiet / medium / loud", fontsize=13)
for ax, label, item in zip(axes, ["Quiet", "Medium", "Loud"],
        [panel_dbs[0], panel_dbs[len(panel_dbs) // 2], panel_dbs[-1]]):
    path, db = item
    librosa.display.specshow(db, sr=SR, hop_length=HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax,
                             cmap="magma", vmin=DB_MIN, vmax=DB_MAX)
    ax.set_title(f"{label}\n{path.name[:30]}", fontsize=9)
plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / "diag_spectrograms.png"), dpi=120)
plt.show()

# ── Plot 2: Global dB histogram ────────────────────────────────────────────
print(f"Computing statistics for {len(sampled_paths)} chunks ...")
all_dbs = np.concatenate([wav_to_logmel(p).ravel() for p in sampled_paths])
pcts    = np.percentile(all_dbs, [1, 5, 50, 95, 99])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_dbs, bins=200, color="steelblue", alpha=0.8)
for p_val, color, lbl in zip(pcts,
        ["red", "orange", "green", "orange", "red"],
        ["p1", "p5", "p50", "p95", "p99"]):
    ax.axvline(p_val, color=color, linestyle="--", linewidth=1.2,
               label=f"{lbl} = {p_val:.1f} dB")
ax.set_title("Global log-Mel dB value distribution")
ax.set_xlabel("dB")
ax.set_ylabel("count")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / "diag_db_histogram.png"), dpi=120)
plt.show()
print("dB percentiles:", {f"p{p}": f"{v:.1f}" for p, v in zip([1, 5, 50, 95, 99], pcts)})

# ── Plot 3: Per-band statistics ────────────────────────────────────────────
mel_stack = np.stack([wav_to_logmel(p) for p in sampled_paths], axis=0)
band_mean = mel_stack.mean(axis=(0, 2))
band_std  = mel_stack.std(axis=(0, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(band_mean)
axes[0].set_title("Mean per Mel bin")
axes[0].set_xlabel("Mel bin")
axes[0].set_ylabel("dB")
axes[1].plot(band_std, color="orange")
axes[1].set_title("Std per Mel bin")
axes[1].set_xlabel("Mel bin")
axes[1].set_ylabel("dB")
fig.suptitle("Per-band statistics (sampled training chunks)")
plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / "diag_per_band_stats.png"), dpi=120)
plt.show()

# ── Plot 4: Normalization comparison ──────────────────────────────────────
sample_db   = wav_to_logmel(sampled_paths[len(sampled_paths) // 2])
norm_01     = (np.clip(sample_db, DB_MIN, DB_MAX) - DB_MIN) / (DB_MAX - DB_MIN)
norm_zscore = (sample_db - band_mean[:, None]) / (band_std[:, None] + 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Normalization comparison — same chunk", fontsize=13)
for ax, data, title, cmap in zip(axes,
        [sample_db, norm_01, norm_zscore],
        ["Raw log-Mel dB", "Fixed-range [0, 1]", "Per-band z-score"],
        ["magma", "viridis", "RdBu_r"]):
    librosa.display.specshow(data, sr=SR, hop_length=HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax, cmap=cmap)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / "diag_normalization_comparison.png"), dpi=120)
plt.show()

print("Diagnostic plots saved to", PROCESSED_DIR)
print("Set NORM_METHOD in Step 8 based on the plots above.")

## Step 8 — Export Tensors for Training

Convert all processed `.wav` chunks to `(128, 188)` log-Mel spectrogram tensors, apply the chosen normalization, group chunks by species into 3-D NumPy arrays, and serialize each to a compressed `.npz` file in `bird_tensors/` on Drive.

A `tensor_index.json` is also written containing chunk counts, normalization metadata, and pre-calculated 70 / 15 / 15 train / validation / test split boundaries per species.

Set `NORM_METHOD = "fixed_range"` (default) or `"per_band_zscore"` (requires Step 7 to have been run first in the same session).

In [ ]:
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 128
SR         = 32_000
DB_MIN     = -80.0
DB_MAX     = 0.0

# ── Choose normalization ──────────────────────────────────────────────────
# "fixed_range"     → clip dB to [DB_MIN, DB_MAX], scale to [0, 1]
# "per_band_zscore" → z-score per Mel bin; requires band_mean / band_std from Step 7
NORM_METHOD = "fixed_range"

TARGET_FRAMES = 188

# ── 5-fold cross-validation ───────────────────────────────────────────────
N_FOLDS   = 5
FOLD_SEED = 42

TENSORS_DIR = Path(DRIVE_ROOT_PATH) / "bird_tensors"
TENSORS_DIR.mkdir(parents=True, exist_ok=True)


def wav_to_tensor(path, norm_params):
    audio, _ = librosa.load(str(path), sr=SR, mono=True)
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel, ref=np.max)

    if norm_params["method"] == "fixed_range":
        lo, hi = norm_params["db_min"], norm_params["db_max"]
        tensor = (np.clip(db, lo, hi) - lo) / (hi - lo)
    elif norm_params["method"] == "per_band_zscore":
        mean   = np.array(norm_params["band_mean"])[:, None]
        std    = np.array(norm_params["band_std"])[:, None] + 1e-6
        tensor = (db - mean) / std
    else:
        tensor = db

    if tensor.shape[1] < TARGET_FRAMES:
        tensor = np.pad(tensor, ((0, 0), (0, TARGET_FRAMES - tensor.shape[1])))
    else:
        tensor = tensor[:, :TARGET_FRAMES]

    return tensor.astype(np.float32)


if NORM_METHOD == "fixed_range":
    norm_params = {"method": "fixed_range", "db_min": DB_MIN, "db_max": DB_MAX}
elif NORM_METHOD == "per_band_zscore":
    norm_params = {
        "method":    "per_band_zscore",
        "band_mean": band_mean.tolist(),
        "band_std":  band_std.tolist(),
    }
else:
    raise ValueError(f"Unknown NORM_METHOD: {NORM_METHOD!r}")

tensor_index = {
    "norm_method": norm_params["method"],
    "norm_params": {k: v for k, v in norm_params.items() if k != "method"},
    "n_folds":     N_FOLDS,
    "fold_seed":   FOLD_SEED,
    "species": {},
}

species_dirs = sorted(d for d in PROCESSED_DIR.iterdir() if d.is_dir())
print(f"Exporting tensors for {len(species_dirs)} species ...")
total_chunks = 0

for species_dir in species_dirs:
    wav_paths = sorted(species_dir.glob("*.wav"))
    if not wav_paths:
        continue

    tensors = []
    for wav_path in wav_paths:
        try:
            tensors.append(wav_to_tensor(wav_path, norm_params))
        except Exception as exc:
            print(f"  skip {wav_path.name}: {exc}")

    if not tensors:
        continue

    arr      = np.stack(tensors, axis=0)
    npz_path = TENSORS_DIR / f"{species_dir.name}.npz"
    np.savez_compressed(str(npz_path), spectrograms=arr)

    n            = len(tensors)
    indices      = list(range(n))
    rng          = random.Random(FOLD_SEED)
    rng.shuffle(indices)
    fold_assignments = [i % N_FOLDS for i in range(n)]
    # distribute shuffled positions into folds
    fold_of_chunk = [None] * n
    for rank, original_idx in enumerate(indices):
        fold_of_chunk[original_idx] = rank % N_FOLDS

    tensor_index["species"][species_dir.name] = {
        "chunk_count":    n,
        "npz_file":       npz_path.name,
        "fold_assignments": fold_of_chunk,
    }
    total_chunks += n
    print(f"  {species_dir.name}: {n} chunks → {npz_path.name}")

index_path = TENSORS_DIR / "tensor_index.json"
with open(index_path, "w", encoding="utf-8") as f:
    json.dump(tensor_index, f, ensure_ascii=False, indent=2)

print(f"\nTensors exported to   : {TENSORS_DIR}")
print(f"Tensor index saved at : {index_path}")
print(f"Total species exported: {len(tensor_index['species'])}")
print(f"Total chunks exported : {total_chunks}")